In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]


In [3]:
data=np.array(data)
label_map={'Wine':0,'Beer':1,'Whiskey':2}
reverse_map={0:'Wine',1:'Beer',2:'Whiskey'}
X=data[:, :3].astype(float)
y=np.array([label_map[label] for label in data[:, 3]])

In [4]:
X

array([[12. ,  1.5,  1. ],
       [ 5. ,  2. ,  0. ],
       [40. ,  0. ,  1. ],
       [13.5,  1.2,  1. ],
       [ 4.5,  1.8,  0. ],
       [38. ,  0.1,  1. ],
       [11.5,  1.7,  1. ],
       [ 5.5,  2.3,  0. ]])

In [5]:
y

array([0, 1, 2, 0, 1, 2, 0, 1])

In [6]:
def gini_impurity(y):
    values,counts=np.unique(y,return_counts=True)
    probs=counts/len(y)
    return (1-np.sum(probs**2))


def entropy(y):
    values,counts=np.unique(y,return_counts=True)
    probs=counts/len(y)
    return (-np.sum(probs*np.log2(probs)))

In [7]:
def best_split_finder(X,y,criterion="gini"):
    all_imp=[]
    for i in range(X.shape[1]):
        vals=np.sort(np.unique(X[:, i]))
        thresholds=(vals[:-1] + vals[1:]) / 2
        for j in thresholds:
            left=[]
            right=[]
            for k in range(len(X)):
                if X[k][i]>=j:
                    right.append(y[k])
                else:
                    left.append(y[k])
            if len(left)==0 or len(right)==0:
                continue
            if criterion=="gini":
                left_imp=gini_impurity(left)
                right_imp=gini_impurity(right)
            else:
                left_imp=entropy(left)
                right_imp=entropy(right)
            all_imp.append([(len(left)/len(X))*left_imp+(len(right)/len(X))*right_imp,i,j])
    all_imp=np.array(all_imp)
    sorted_arr=all_imp[all_imp[:,0].argsort()]
    return int(sorted_arr[0][1]),int(sorted_arr[0][2])
                

In [8]:
class Node:
    def __init__(self,feature_index,threshold,left,right,value):
        self.feature_index=feature_index
        self.threshold=threshold
        self.left=left
        self.right=right
        self.value=value

In [9]:
def build_tree(X,y,depth=None,max_depth=None,min_samples_split=None):
    
    if gini_impurity(y)==0:
        return Node(None,None,None,None,y[0])
    if depth>=max_depth or len(y)<min_samples_split:
        values,counts=np.unique(y, return_counts=True)
        value=values[np.argmax(counts)]
        return Node(None,None,None,None,value)
    else:
        i,j=best_split_finder(X,y)
        X_left=[]
        X_right=[]
        y_left=[]
        y_right=[]
        for k in range(len(X)):
            if X[k][i]>=j:
                X_right.append(X[k])
                y_right.append(y[k])
            else:
                X_left.append(X[k])
                y_left.append(y[k])
        X_left=np.array(X_left)
        X_right=np.array(X_right)
        y_left=np.array(y_left)
        y_right=np.array(y_right)
        if len(y_left)==0 or len(y_right)==0:
            values,counts=np.unique(y,return_counts=True)
            value=values[np.argmax(counts)]
            return Node(None,None,None,None,value)
        left=build_tree(X_left,y_left,depth+1,max_depth,min_samples_split)
        right=build_tree(X_right,y_right,depth+1,max_depth,min_samples_split)
        return Node(i,j,left,right,None)
        
    
            
    

In [10]:
root=build_tree(X,y,depth=0,max_depth=5,min_samples_split=2)

In [11]:
root

In [12]:
feature_names = ["Alcohol", "Sugar", "Color"]

In [13]:
def predict_one(root,new_sample):
    if root.value!=None:
        return root.value
    else:
        if new_sample[root.feature_index]>=root.threshold:
            return predict_one(root.right,new_sample)
        else:
            return predict_one(root.left,new_sample)
        

In [14]:
def predict(test_data):
    a=[]
    for ele in test_data:
        a.append(reverse_map[predict_one(root,ele)])
    return a

In [15]:
test_data = np.array([
    [6.0, 2.1, 0],   # Expected: Beer
    [39.0, 0.05, 1], # Expected: Whiskey
    [13.0, 1.3, 1]   # Expected: Wine
])


In [16]:
predict(test_data)

['Beer', 'Whiskey', 'Wine']

In [17]:
print(root.feature_index, root.threshold, root.value)

0 8 None


In [18]:
def print_tree(node,depth):
    indent="  " * depth
    if node.value==None:
        print(indent+"if:"+feature_names[node.feature_index]+"<"+str(node.threshold)+"->go left")
        print_tree(node.left,depth+1)
        print(indent+"else:"+"->go right")
        print_tree(node.right,depth+1)
    else:
        print(indent+reverse_map[node.value])
        
    

In [19]:
print_tree(root,depth=0)

if:Alcohol<8->go left
  Beer
else:->go right
  if:Alcohol<25->go left
    Wine
  else:->go right
    Whiskey
